In [ ]:
# Proyecto El Detective de Señales - Semana 1 (Fase 1)
# Aguilar Vargas Evelyn Adriana - 5402
# Fundamentos de Telecomunicaciones

# ===================== FASE 1: GENERACION Y PREPARACION DE DATOS =====================

# 1.1 Librerias
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.model_selection import train_test_split

In [ ]:
# 1.2 Parametros de la señal
# Con fs = 16 kHz se cumple Nyquist (fs >= 2*fmax), la frecuencia mas alta es de unos 3.5 kHz
fs = 16000      # frecuencia de muestreo (Hz)
fc = 2000       # frecuencia de la portadora (Hz)
Rs = 1000       # tasa de simbolos (baudios)
muestras_simbolo = fs // Rs          # 16 muestras por simbolo
num_simbolos = 16                    # simbolos por señal
N = muestras_simbolo * num_simbolos  # 256 muestras por señal

clases = ["ASK", "FSK", "PSK", "QAM"]

print("muestras por simbolo:", muestras_simbolo)
print("muestras por señal:", N)

In [ ]:
# 1.3 Funciones para generar las señales
# A cada señal se le da una fase inicial al azar porque el receptor no sabe en que fase llega la portadora

t = np.arange(muestras_simbolo) / fs   # tiempo de un simbolo

# ASK: bit 1 = se manda la portadora, bit 0 = amplitud cero
def ask(fase):
    bits = np.random.randint(0, 2, num_simbolos)
    s = []
    for k in range(num_simbolos):
        tk = t + k / Rs
        s.append(bits[k] * np.cos(2*np.pi*fc*tk + fase))
    return np.concatenate(s)

# FSK: bit 0 = 1500 Hz, bit 1 = 2500 Hz
def fsk(fase):
    bits = np.random.randint(0, 2, num_simbolos)
    s = []
    for k in range(num_simbolos):
        tk = t + k / Rs
        if bits[k] == 1:
            f = fc + 500
        else:
            f = fc - 500
        s.append(np.cos(2*np.pi*f*tk + fase))
    return np.concatenate(s)

# PSK (BPSK): bit 1 = fase 0°, bit 0 = fase 180°
def psk(fase):
    bits = np.random.randint(0, 2, num_simbolos)
    s = []
    for k in range(num_simbolos):
        tk = t + k / Rs
        if bits[k] == 1:
            s.append(np.cos(2*np.pi*fc*tk + fase))
        else:
            s.append(np.cos(2*np.pi*fc*tk + fase + np.pi))
    return np.concatenate(s)

# 16-QAM: cambia amplitud y fase, I y Q pueden valer -3, -1, 1 o 3
# son 4x4 = 16 combinaciones, o sea 4 bits por simbolo
def qam(fase):
    niveles = [-3, -1, 1, 3]
    s = []
    for k in range(num_simbolos):
        I = np.random.choice(niveles)
        Q = np.random.choice(niveles)
        tk = t + k / Rs
        s.append(I*np.cos(2*np.pi*fc*tk + fase) - Q*np.sin(2*np.pi*fc*tk + fase))
    return np.concatenate(s)

# normalizar para que todas las señales tengan potencia 1
def normalizar(x):
    return x / np.sqrt(np.mean(x**2))

generadores = [ask, fsk, psk, qam]

In [ ]:
# Grafica de un ejemplo de cada modulacion sin ruido
np.random.seed(7)
tiempo = np.arange(N) / fs * 1000   # en ms

plt.figure(figsize=(12, 8))
for i in range(4):
    plt.subplot(4, 1, i+1)
    plt.plot(tiempo, normalizar(generadores[i](0)))
    plt.ylabel(clases[i])
    plt.grid()
plt.xlabel("tiempo (ms)")
plt.tight_layout()
plt.show()

In [ ]:
# Espectro de cada modulacion (promedio de 30 señales)
# FSK debe tener dos picos (1.5 y 2.5 kHz) y las demas estar centradas en 2 kHz
plt.figure(figsize=(10, 4))
for i in range(4):
    P_total = 0
    for j in range(30):
        x = normalizar(generadores[i](np.random.uniform(0, 2*np.pi)))
        f, P = signal.welch(x, fs=fs, nperseg=128)
        P_total = P_total + P
    plt.plot(f, 10*np.log10(P_total/30), label=clases[i])
plt.xlabel("frecuencia (Hz)")
plt.ylabel("potencia (dB)")
plt.xlim(0, 6000)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 1.5 Ruido AWGN
# la potencia del ruido se calcula con: P_ruido = P_señal / 10^(SNR/10)
def agregar_ruido(x, snr):
    p_senal = np.mean(x**2)
    p_ruido = p_senal / (10**(snr/10))
    ruido = np.random.normal(0, np.sqrt(p_ruido), len(x))
    return x + ruido

# misma señal QAM con diferentes niveles de ruido
# con 20 dB casi no se nota, con -10 dB la señal casi no se ve
np.random.seed(3)
x = normalizar(qam(0))
snrs_prueba = [20, 10, 0, -10]

plt.figure(figsize=(12, 7))
for i in range(4):
    plt.subplot(4, 1, i+1)
    plt.plot(tiempo, agregar_ruido(x, snrs_prueba[i]), "r", linewidth=0.8)
    plt.plot(tiempo, x, "b", linewidth=0.8)
    plt.ylabel(str(snrs_prueba[i]) + " dB")
    plt.grid()
plt.xlabel("tiempo (ms)")
plt.tight_layout()
plt.show()

In [ ]:
# 1.4 Generacion del dataset
# 7 niveles de SNR x 300 señales x 4 modulaciones = 8400 señales (2100 por modulacion)
# X = señales, y = etiqueta (0=ASK, 1=FSK, 2=PSK, 3=QAM), snr = nivel de ruido de cada señal
np.random.seed(42)
lista_snr = [-10, -5, 0, 5, 10, 15, 20]

X = []
y = []
snr = []
for etiqueta in range(4):
    for s in lista_snr:
        for n in range(300):
            fase = np.random.uniform(0, 2*np.pi)
            x = normalizar(generadores[etiqueta](fase))
            x = normalizar(agregar_ruido(x, s))
            X.append(x)
            y.append(etiqueta)
            snr.append(s)

X = np.array(X)
y = np.array(y)
snr = np.array(snr)

print("tamaño de X:", X.shape)
for i in range(4):
    print(clases[i], ":", np.sum(y == i), "señales")

In [ ]:
# 1.6 Separacion: 80% entrenamiento y 20% prueba
# stratify=y para que haya la misma cantidad de cada modulacion en los dos grupos
X_train, X_test, y_train, y_test, snr_train, snr_test = train_test_split(
    X, y, snr, test_size=0.2, random_state=42, stratify=y)

print("entrenamiento:", X_train.shape[0])
print("prueba:", X_test.shape[0])

# guardar los datos para usarlos despues
np.savez("dataset.npz", X_train=X_train, X_test=X_test, y_train=y_train,
         y_test=y_test, snr_train=snr_train, snr_test=snr_test)